# 3.3 Annotation Ceiling

Estimate inter-annotation agreement for UdonPred evaluation datasets that overlap at protein or residue level. This is an annotation agreement estimate, not model performance.


## Method

This notebook estimates an experimental annotation ceiling by comparing labels from different UdonPred evaluation datasets on proteins that appear in both datasets. The exact ceiling uses exact ID or exact sequence matches. If MMseqs2 is installed, the notebook also evaluates local-alignment ceilings at multiple identity thresholds. High agreement suggests that a model could, in principle, transfer between those disorder definitions. Low agreement suggests that the datasets encode different disorder concepts, contain annotation noise, or have limited overlap.

This is not model performance. It is an inter-annotation agreement estimate and should be interpreted as an approximate upper-bound diagnostic for cross-dataset transfer.


In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RESULTS = ROOT / "results"

exact_ceiling_dir = RESULTS / "annotation_ceiling"
mmseqs_ceiling_dir = RESULTS / "annotation_ceiling_mmseqs"
RUN_MMSEQS = shutil.which("mmseqs") is not None
MMSEQS_IDENTITIES = [100, 98, 95, 90, 85, 80]
MMSEQS_MIN_COVERAGE = 0.8

def run_ceiling(output_dir, extra_args=None):
    summary_csv = output_dir / "annotation_ceiling_summary.csv"
    overlap_csv = output_dir / "overlap_details.csv"
    if summary_csv.exists() and overlap_csv.exists():
        print(f"Annotation ceiling results already exist in {output_dir}, skipping computation.")
        return summary_csv, overlap_csv

    print(f"Running annotation ceiling computation into {output_dir}...")
    output_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        str(ROOT / "scripts" / "estimate_annotation_ceiling.py"),
        "--udonpred-dir",
        str(ROOT / "UdonPred"),
        "--output-dir",
        str(output_dir),
    ]
    if extra_args:
        cmd.extend(extra_args)
    result = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    result.check_returncode()
    return summary_csv, overlap_csv

def load_ceiling_result(output_dir, source):
    summary_csv = output_dir / "annotation_ceiling_summary.csv"
    overlap_csv = output_dir / "overlap_details.csv"
    summary = pd.read_csv(summary_csv)
    details = pd.read_csv(overlap_csv)
    for frame in [summary, details]:
        if "comparison_level" not in frame.columns:
            frame["comparison_level"] = "exact"
    for column in ["min_identity", "min_coverage"]:
        if column not in summary.columns:
            summary[column] = np.nan
    for column in ["percent_identity", "query_coverage", "target_coverage", "alignment_length"]:
        if column not in details.columns:
            details[column] = np.nan
    summary["source"] = source
    details["source"] = source
    return summary, details

run_ceiling(exact_ceiling_dir)

if RUN_MMSEQS:
    run_ceiling(
        mmseqs_ceiling_dir,
        [
            "--use-mmseqs",
            "--mmseqs-identities",
            *[str(value) for value in MMSEQS_IDENTITIES],
            "--mmseqs-min-coverage",
            str(MMSEQS_MIN_COVERAGE),
        ],
    )
elif (mmseqs_ceiling_dir / "annotation_ceiling_summary.csv").exists():
    print("MMseqs is not installed, but existing MMseqs ceiling results will be loaded.")
else:
    print("MMseqs is not installed; notebook will evaluate exact-match ceilings only.")

summary_frames = []
detail_frames = []
for output_dir, source in [(exact_ceiling_dir, "exact_run"), (mmseqs_ceiling_dir, "mmseqs_run")]:
    if (output_dir / "annotation_ceiling_summary.csv").exists() and (output_dir / "overlap_details.csv").exists():
        summary, details = load_ceiling_result(output_dir, source)
        summary_frames.append(summary)
        detail_frames.append(details)

ceiling_summary = pd.concat(summary_frames, ignore_index=True)
overlap_details = pd.concat(detail_frames, ignore_index=True)

ceiling_summary["pair"] = ceiling_summary["dataset_a"] + " vs " + ceiling_summary["dataset_b"]
overlap_details["pair"] = overlap_details["dataset_a"] + " vs " + overlap_details["dataset_b"]
ceiling_summary["level_label"] = ceiling_summary["source"] + ":" + ceiling_summary["comparison_level"]
overlap_details["level_label"] = overlap_details["source"] + ":" + overlap_details["comparison_level"]

available_levels = ceiling_summary[["source", "comparison_level"]].drop_duplicates().sort_values(["source", "comparison_level"])
available_levels


In [ ]:
dataset_order = ["trizod", "chezod", "softdis", "pdbflex", "atlas", "plddt", "disprot"]

selected_source = "mmseqs_run" if "mmseqs_run" in set(ceiling_summary["source"]) else "exact_run"
available_selected_levels = set(ceiling_summary.loc[ceiling_summary["source"] == selected_source, "comparison_level"])
if selected_source == "mmseqs_run" and "mmseqs_95" in available_selected_levels:
    selected_comparison_level = "mmseqs_95"
elif selected_source == "mmseqs_run":
    selected_comparison_level = sorted(available_selected_levels)[0]
else:
    selected_comparison_level = "exact"
selected_summary = ceiling_summary[
    (ceiling_summary["source"] == selected_source)
    & (ceiling_summary["comparison_level"] == selected_comparison_level)
].copy()

all_pairs = selected_summary.drop_duplicates(["dataset_a", "dataset_b", "comparison_level"])[
    ["dataset_a", "dataset_b", "comparison_level", "match_mode", "n_proteins_overlap", "n_residues_compared"]
]

residue_overlap = pd.DataFrame(np.nan, index=dataset_order, columns=dataset_order)
protein_overlap = pd.DataFrame(np.nan, index=dataset_order, columns=dataset_order)
match_modes = pd.DataFrame("", index=dataset_order, columns=dataset_order)

for row in all_pairs.itertuples(index=False):
    residue_overlap.loc[row.dataset_a, row.dataset_b] = row.n_residues_compared
    residue_overlap.loc[row.dataset_b, row.dataset_a] = row.n_residues_compared
    protein_overlap.loc[row.dataset_a, row.dataset_b] = row.n_proteins_overlap
    protein_overlap.loc[row.dataset_b, row.dataset_a] = row.n_proteins_overlap
    match_modes.loc[row.dataset_a, row.dataset_b] = row.match_mode
    match_modes.loc[row.dataset_b, row.dataset_a] = row.match_mode

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(
    protein_overlap,
    annot=True,
    fmt=".0f",
    cmap="mako",
    mask=protein_overlap.isna(),
    ax=axes[0],
)
axes[0].set_title(f"Overlapping Proteins ({selected_source}:{selected_comparison_level})")
sns.heatmap(
    residue_overlap,
    annot=True,
    fmt=".0f",
    cmap="mako",
    mask=residue_overlap.isna(),
    ax=axes[1],
)
axes[1].set_title(f"Aligned Residues Compared ({selected_source}:{selected_comparison_level})")
plt.tight_layout()
plt.show()

all_pairs.sort_values("n_residues_compared", ascending=False).head(10)


## Overlap Caution

Agreement estimates are only meaningful when enough residues overlap. Pairs with very few overlapping residues should be interpreted cautiously, even if their agreement metric is high.


In [ ]:
small_overlap_pairs = all_pairs.sort_values("n_residues_compared").head(10)
small_overlap_pairs


## Primary Agreement Metric

For continuous-continuous dataset pairs, the primary agreement metric is Spearman correlation. For pairs involving DisProt, which is binary, the primary agreement metric is AUROC. These choices mirror the evaluation style used in the UdonPred 7x7 matrix: continuous test datasets use rank correlation, while DisProt uses binary ranking metrics.


In [ ]:
primary_metrics = {
    "continuous-continuous": "spearman",
    "continuous-binary": "auroc",
    "binary-continuous": "auroc",
    "binary-binary": "mcc",
}

primary_rows = []
for row in all_pairs.itertuples(index=False):
    pair_metrics = selected_summary[
        (selected_summary["dataset_a"] == row.dataset_a)
        & (selected_summary["dataset_b"] == row.dataset_b)
        & (selected_summary["comparison_level"] == row.comparison_level)
    ].copy()
    if pair_metrics.empty:
        continue
    type_key = f"{pair_metrics.iloc[0]['annotation_type_a']}-{pair_metrics.iloc[0]['annotation_type_b']}"
    metric = primary_metrics.get(type_key, "spearman")
    metric_row = pair_metrics[pair_metrics["metric"] == metric]
    if metric_row.empty:
        value = np.nan
    else:
        value = metric_row.iloc[0]["value"]
    primary_rows.append(
        {
            "dataset_a": row.dataset_a,
            "dataset_b": row.dataset_b,
            "comparison_level": row.comparison_level,
            "metric": metric,
            "value": value,
            "n_residues_compared": row.n_residues_compared,
            "match_mode": row.match_mode,
        }
    )

primary_agreement = pd.DataFrame(primary_rows)
agreement_matrix = pd.DataFrame(np.nan, index=dataset_order, columns=dataset_order)
for row in primary_agreement.itertuples(index=False):
    agreement_matrix.loc[row.dataset_a, row.dataset_b] = row.value
    agreement_matrix.loc[row.dataset_b, row.dataset_a] = row.value

plt.figure(figsize=(8, 6))
sns.heatmap(
    agreement_matrix,
    annot=True,
    fmt=".3f",
    cmap="viridis",
    vmin=-1,
    vmax=1,
    mask=agreement_matrix.isna(),
)
plt.title(f"Primary Annotation Agreement ({selected_source}:{selected_comparison_level})\nSpearman for continuous pairs, AUROC for DisProt pairs")
plt.show()

primary_agreement.sort_values("n_residues_compared", ascending=False)


## Compare Exact And MMseqs Ceiling Levels

When MMseqs results are available, the tables and plots below compare the exact ceiling with progressively looser local-alignment identity thresholds. More permissive thresholds usually increase overlap, but they can also compare less equivalent protein regions, so agreement changes should be interpreted together with residue coverage and overlap size.


In [ ]:
def primary_metric_for_types(type_a, type_b):
    return primary_metrics.get(f"{type_a}-{type_b}", "spearman")

level_primary_rows = []
level_pairs = ceiling_summary.drop_duplicates(
    ["source", "comparison_level", "dataset_a", "dataset_b"]
)[[
    "source",
    "comparison_level",
    "dataset_a",
    "dataset_b",
    "match_mode",
    "n_proteins_overlap",
    "n_residues_compared",
]]

for row in level_pairs.itertuples(index=False):
    pair_metrics = ceiling_summary[
        (ceiling_summary["source"] == row.source)
        & (ceiling_summary["comparison_level"] == row.comparison_level)
        & (ceiling_summary["dataset_a"] == row.dataset_a)
        & (ceiling_summary["dataset_b"] == row.dataset_b)
    ].copy()
    if pair_metrics.empty:
        continue
    metric = primary_metric_for_types(
        pair_metrics.iloc[0]["annotation_type_a"],
        pair_metrics.iloc[0]["annotation_type_b"],
    )
    metric_row = pair_metrics[pair_metrics["metric"] == metric]
    value = np.nan if metric_row.empty else metric_row.iloc[0]["value"]
    level_primary_rows.append(
        {
            "source": row.source,
            "comparison_level": row.comparison_level,
            "level_label": f"{row.source}:{row.comparison_level}",
            "dataset_a": row.dataset_a,
            "dataset_b": row.dataset_b,
            "pair": f"{row.dataset_a} vs {row.dataset_b}",
            "metric": metric,
            "value": value,
            "n_proteins_overlap": row.n_proteins_overlap,
            "n_residues_compared": row.n_residues_compared,
            "match_mode": row.match_mode,
        }
    )

primary_agreement_by_level = pd.DataFrame(level_primary_rows)
primary_agreement_by_level.sort_values(
    ["pair", "source", "comparison_level"], ascending=[True, True, False]
).head(30)


In [ ]:
plot_levels = primary_agreement_by_level[
    primary_agreement_by_level["value"].notna()
    & primary_agreement_by_level["comparison_level"].astype(str).str.startswith("mmseqs_")
].copy()

if plot_levels.empty:
    print("No MMseqs comparison levels available. Install MMseqs2 and rerun the first cell to generate them.")
else:
    plot_levels["identity_threshold"] = plot_levels["comparison_level"].str.replace("mmseqs_", "", regex=False).astype(float)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.lineplot(
        data=plot_levels,
        x="identity_threshold",
        y="n_residues_compared",
        hue="pair",
        marker="o",
        ax=axes[0],
    )
    axes[0].invert_xaxis()
    axes[0].set_title("Residue Overlap Across MMseqs Identity Thresholds")
    axes[0].set_xlabel("Minimum identity (%)")
    axes[0].set_ylabel("Residues compared")

    sns.lineplot(
        data=plot_levels,
        x="identity_threshold",
        y="value",
        hue="pair",
        style="metric",
        marker="o",
        ax=axes[1],
    )
    axes[1].invert_xaxis()
    axes[1].axhline(0, color="black", linewidth=0.8)
    axes[1].set_title("Primary Agreement Across MMseqs Identity Thresholds")
    axes[1].set_xlabel("Minimum identity (%)")
    axes[1].set_ylabel("Primary agreement")

    for ax in axes:
        ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


## Compare UdonPred To The Estimated Ceiling

The table below compares UdonPred transfer scores with the primary annotation agreement for the same dataset pair where the metric is comparable. This is a rough diagnostic, not a strict mathematical bound, because the ceiling is estimated only on overlapping proteins/residues while the UdonPred matrix is evaluated on full test sets.

For pairs involving DisProt, only directions where DisProt is the test dataset are directly comparable to the AUROC ceiling metric.


In [ ]:
udon_matrix_path = RESULTS / "udonpred_matrix" / "matrix.csv"
if not udon_matrix_path.exists():
    raise FileNotFoundError(
        f"Missing {udon_matrix_path}. Run the UdonPred matrix notebook/script first."
    )

udon_matrix = pd.read_csv(udon_matrix_path).set_index("train_dataset")

def udon_column_for_test_dataset(test_dataset):
    if test_dataset == "disprot":
        return "disprot\n(AUROC)"
    return test_dataset

def udon_metric_for_test_dataset(test_dataset):
    if test_dataset == "disprot":
        return "auroc"
    return "spearman"

comparison_rows = []
for row in primary_agreement.itertuples(index=False):
    for train_dataset, test_dataset in [
        (row.dataset_a, row.dataset_b),
        (row.dataset_b, row.dataset_a),
    ]:
        test_metric = udon_metric_for_test_dataset(test_dataset)
        if test_metric != row.metric:
            continue
        column = udon_column_for_test_dataset(test_dataset)
        udon_value = udon_matrix.loc[train_dataset, column]
        comparison_rows.append(
            {
                "train_dataset": train_dataset,
                "test_dataset": test_dataset,
                "ceiling_pair": f"{row.dataset_a} vs {row.dataset_b}",
                "metric": row.metric,
                "udon_score": udon_value,
                "annotation_ceiling": row.value,
                "gap_to_ceiling": row.value - udon_value,
                "n_residues_compared": row.n_residues_compared,
                "match_mode": row.match_mode,
            }
        )

udon_vs_ceiling = pd.DataFrame(comparison_rows).sort_values(
    ["metric", "gap_to_ceiling"], ascending=[True, True]
)
udon_vs_ceiling


In [ ]:
metric_plot = selected_summary[
    selected_summary["metric"].isin(["spearman", "pearson", "auroc", "average_precision", "f1_thresholded"])
    & selected_summary["value"].notna()
].copy()
metric_plot["comparison"] = metric_plot["dataset_a"] + " vs " + metric_plot["dataset_b"]
metric_plot = metric_plot.sort_values(["metric", "value"], ascending=[True, False])

plt.figure(figsize=(12, 6))
sns.barplot(data=metric_plot, x="value", y="comparison", hue="metric")
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Metric value")
plt.ylabel("Dataset pair")
plt.title(f"Annotation Agreement Metrics ({selected_source}:{selected_comparison_level})")
plt.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
ceiling_summary[
    ceiling_summary["notes"].notna() & (ceiling_summary["notes"] != "")
][["source", "comparison_level", "dataset_a", "dataset_b", "metric", "value", "notes"]].drop_duplicates().head(20)


## Interpretation Notes

After running the notebook, summarize the result in terms of dataset agreement rather than model accuracy:

- Strongest annotation agreement: fill in from `primary_agreement.sort_values("value", ascending=False)`.
- Weakest annotation agreement: fill in from `primary_agreement.sort_values("value")`.
- Pairs with low agreement should not be expected to transfer well, even for a strong model.
- Pairs with small overlap should be treated as low-confidence ceiling estimates.
- Negative `gap_to_ceiling` values in the comparison table mean UdonPred exceeds this approximate overlap-based ceiling estimate; this usually indicates that the estimate is noisy, based on limited overlap, or not directly comparable to the full-test-set UdonPred score.
